# 📊 RevertIQ — Feature Engineering

This notebook demonstrates:
1. Computing technical indicators for each stock
2. Computing cross-sectional features across all stocks
3. Visualising indicator distributions
4. Understanding the mean reversion score

In [ ]:
import warnings; warnings.filterwarnings('ignore')

from revertiq.config.settings import get_default_config
from revertiq.data.cleaner import DataCleaner
from revertiq.data.downloader import DataDownloader
from revertiq.indicators.technical import TechnicalIndicators
from revertiq.features.engineering import FeatureEngineer

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

config = get_default_config()
print("Ready ✓")

## 1. Load Clean Data

In [ ]:
# Load processed data (run 01_data_pipeline first)
cleaner = DataCleaner(config.data)
try:
    clean_data = cleaner.load_all_processed()
    combined = cleaner.get_combined_df()
    print(f"Loaded {len(clean_data)} tickers")
except FileNotFoundError:
    print("No processed data found. Run 01_data_pipeline.ipynb first.")
    print("Downloading fresh data...")
    dl = DataDownloader(config.data)
    dl.download_all()
    clean_data = cleaner.clean_all()
    combined = cleaner.get_combined_df()

# Load NIFTY index
dl = DataDownloader(config.data)
index_data = dl.download_index_data()
nifty_data = index_data.get('nifty50', pd.DataFrame())
if not nifty_data.empty:
    nifty_data.columns = [c.lower() for c in nifty_data.columns]

## 2. Technical Indicators

In [ ]:
# Compute indicators for all stocks
ti = TechnicalIndicators()
enriched_dfs = []

for ticker in combined['ticker'].unique():
    stock_df = combined[combined['ticker'] == ticker].copy().sort_values('date')
    enriched = ti.compute_all(stock_df, config.indicator)
    enriched_dfs.append(enriched)

stock_data = pd.concat(enriched_dfs, ignore_index=True)
indicator_cols = [c for c in stock_data.columns if c not in combined.columns]
print(f"New indicator columns ({len(indicator_cols)}):")
for c in indicator_cols:
    print(f"  • {c}")

In [ ]:
# Visualise RSI distribution
latest = stock_data.groupby('ticker').tail(1)
fig = px.histogram(latest, x='rsi', nbins=50,
                   title='RSI(2) Distribution (Latest Day)',
                   color_discrete_sequence=['#58a6ff'])
fig.add_vline(x=10, line_dash='dash', line_color='#3fb950',
              annotation_text='Buy threshold (RSI<10)')
fig.update_layout(template='plotly_dark',
                  paper_bgcolor='#161b22', plot_bgcolor='#0d1117')
fig.show()

In [ ]:
# Visualise Z-score distribution
fig = px.histogram(latest, x='zscore', nbins=50,
                   title='Z-score(20) Distribution (Latest Day)',
                   color_discrete_sequence=['#bc8cff'])
fig.add_vline(x=-1.5, line_dash='dash', line_color='#3fb950',
              annotation_text='Buy threshold (Z<-1.5)')
fig.add_vline(x=0, line_dash='dot', line_color='#8b949e')
fig.update_layout(template='plotly_dark',
                  paper_bgcolor='#161b22', plot_bgcolor='#0d1117')
fig.show()

In [ ]:
# Single stock indicator deep-dive
sample = 'RELIANCE.NS'
rel = stock_data[stock_data['ticker'] == sample].copy().set_index('date').last('6M')

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    row_heights=[0.5, 0.25, 0.25],
                    subplot_titles=[f'{sample} Price + Bollinger', 'RSI(2)', 'Z-score(20)'])

fig.add_trace(go.Scatter(x=rel.index, y=rel['close'], name='Close',
                         line=dict(color='#58a6ff')), row=1, col=1)
if 'bb_upper' in rel.columns:
    fig.add_trace(go.Scatter(x=rel.index, y=rel['bb_upper'], name='BB Upper',
                             line=dict(color='#8b949e', dash='dash')), row=1, col=1)
    fig.add_trace(go.Scatter(x=rel.index, y=rel['bb_lower'], name='BB Lower',
                             fill='tonexty', fillcolor='rgba(139,148,158,0.08)',
                             line=dict(color='#8b949e', dash='dash')), row=1, col=1)

if 'rsi' in rel.columns:
    fig.add_trace(go.Scatter(x=rel.index, y=rel['rsi'], name='RSI(2)',
                             line=dict(color='#d29922')), row=2, col=1)
    fig.add_hline(y=10, line_dash='dash', line_color='#3fb950', row=2, col=1)
    fig.add_hline(y=70, line_dash='dash', line_color='#f85149', row=2, col=1)

if 'zscore' in rel.columns:
    fig.add_trace(go.Scatter(x=rel.index, y=rel['zscore'], name='Z-score',
                             line=dict(color='#bc8cff')), row=3, col=1)
    fig.add_hline(y=-1.5, line_dash='dash', line_color='#3fb950', row=3, col=1)
    fig.add_hline(y=0, line_dash='dot', line_color='#8b949e', row=3, col=1)

fig.update_layout(template='plotly_dark', height=700,
                  paper_bgcolor='#161b22', plot_bgcolor='#0d1117',
                  title=dict(text=f'{sample} — Indicator Deep Dive', font=dict(color='#58a6ff')))
fig.show()

## 3. Cross-Sectional Features

In [ ]:
fe = FeatureEngineer(config.indicator, config.ranking)
features = fe.compute_all_features(stock_data, nifty_data)

new_cols = [c for c in features.columns if c not in stock_data.columns]
print(f"Cross-sectional features added ({len(new_cols)}):")
for c in new_cols:
    print(f"  • {c}")

In [ ]:
# Cross-sectional percentile heatmap (latest day)
pct_cols = [c for c in features.columns if c.startswith('cs_percentile')]
if pct_cols:
    latest_f = features.groupby('ticker').tail(1).set_index('ticker')[pct_cols]
    latest_f = latest_f.sort_values(pct_cols[0]).head(20)
    
    fig = px.imshow(latest_f, text_auto='.0f',
                    title='Cross-Sectional Percentile (Bottom 20 — Buy Candidates)',
                    color_continuous_scale='RdYlGn_r', aspect='auto')
    fig.update_layout(template='plotly_dark',
                      paper_bgcolor='#161b22', plot_bgcolor='#0d1117')
    fig.show()

In [ ]:
# Mean reversion score distribution
if 'mean_reversion_score' in features.columns:
    latest_mr = features.groupby('ticker').tail(1)
    fig = px.histogram(latest_mr, x='mean_reversion_score', nbins=30,
                       title='Mean Reversion Score Distribution',
                       color_discrete_sequence=['#39d353'])
    fig.update_layout(template='plotly_dark',
                      paper_bgcolor='#161b22', plot_bgcolor='#0d1117')
    fig.show()
    
    # Top candidates
    top = latest_mr.nlargest(10, 'mean_reversion_score')[['ticker', 'mean_reversion_score', 'rsi', 'zscore']]
    print("\nTop 10 Mean Reversion Candidates:")
    print(top.to_string(index=False))